## 4. Givens rotations: the residual for free (`gmres_v3`)

*Same math, computed better.* Like Steps 2 and 3, this step changes how GMRES computes its answer, not what the answer is. In exact arithmetic the new solver produces exactly the same iterates as `gmres_v2`. In floating point they should agree up to round-off, and we'll check that they do.

**Where Step 3 left off.** We want to solve $Ax = b$, where $A$ is a nonsingular $n \times n$ matrix that we can only apply to vectors, through a function `matvec` $: v \mapsto Av$. Starting from $x_0 = 0$, GMRES picks at step $k$ the vector $x_k$ in the Krylov space

$$
\mathcal{K}_k = \operatorname{span}\{\, b,\ Ab,\ \dots,\ A^{k-1} b \,\}
$$

whose residual $r_k = b - A x_k$ has the smallest 2-norm, and stops once $\|r_k\|_2 \le \text{rtol} \cdot \|b\|_2$.

The Arnoldi process (Step 2) builds an orthonormal basis $v_1, v_2, \dots$ of $\mathcal{K}_k$, one matvec per step, starting from $v_1 = b / \beta$ with $\beta = \|b\|_2$. The numbers $h_{ij}$ it computes along the way fill the $(k+1) \times k$ upper Hessenberg matrix $\bar H_k$ (zero below its first subdiagonal), and they satisfy the Arnoldi relation $A V_k = V_{k+1} \bar H_k$, where $V_k = [\, v_1 \mid \cdots \mid v_k \,]$. Step 3 used that relation to write $x_k = V_k\, y$, with a weight vector $y \in \mathbb{R}^k$, and to shrink GMRES to the small least-squares problem

$$
\min_{y \in \mathbb{R}^k} \big\| \beta e_1 - \bar H_k\, y \big\|_2 ,
\qquad e_1 = (1, 0, \dots, 0) \in \mathbb{R}^{k+1} ,
$$

whose minimum is $\|r_k\|_2$.

`gmres_v2` hands that problem to `np.linalg.lstsq`, from scratch, at every step. From each solve it needs one number, the residual norm, to decide whether to stop; the $y$ it computes is thrown away at every step but the last. Yet $\bar H_k$ is just $\bar H_{k-1}$ with one column added on the right, plus a bottom row that is zero except for that column's entry $h_{k+1,k}$.

**The plan for this step.** Turn $\bar H_k$ into an upper-triangular matrix with a sequence of small rotations, one per column, so that a new column needs only the rotations already found plus one new one. The residual norm then falls out of the update as a single number, with no least-squares solve and no $y$. The weights $y$ are computed once, at the end, by back-substitution. That is `gmres_v3`.

### Setup

[tutorial_utils.py](tutorial_utils.py) holds what every step of the tutorial uses: `CountingMatvec` from Step 1, the example problems and the plot style. Each example builder returns a pair `(A, b)`. As before, a solver only ever gets the matrix wrapped in a `CountingMatvec`; we use `A` directly only to *check* answers.

The example problems in this notebook, each a matrix $A$ with a right-hand side $b$:

- **`geo3`**: the 3×3 running example from Steps 1–3,
  $$
  A = \begin{pmatrix} 3 & 1 & 0 \\ 0 & 2 & 1 \\ 1 & 0 & 2 \end{pmatrix}, \qquad b = (1, 1, 1) .
  $$
  Nonsymmetric and well conditioned, and small enough to check every number by hand.
- **`diag_narrow`**: the 100×100 diagonal matrix $A = \operatorname{diag}(\lambda_1, \dots, \lambda_{100})$ with eigenvalues evenly spaced from 1 to 1.5, $\lambda_i = 1 + 0.5\,(i-1)/99$, and $b = (1, \dots, 1)$. GMRES solves it in 11 steps.
- **`diag_spread`**: the 20×20 diagonal matrix $A = \operatorname{diag}(\lambda_1, \dots, \lambda_{20})$ with eigenvalues evenly spaced on a log scale from 1 to $10^4$, $\lambda_i = 10^{4(i-1)/19}$, and $b = (1, \dots, 1)$. GMRES needs all 20 steps.
- **`convdiff_n`**: the 20×20 tridiagonal matrix from Step 3, section 3.3. It discretises $-u''(s) + c\, u'(s) = 1$ on $[0, 1]$ with $u(0) = u(1) = 0$, on 20 interior grid points with spacing $\Delta s = 1/21$:
  $$
  \frac{2}{\Delta s^2} \text{ on the diagonal}, \qquad
  -\frac{1}{\Delta s^2} - \frac{c}{2 \Delta s} \text{ below it}, \qquad
  -\frac{1}{\Delta s^2} + \frac{c}{2 \Delta s} \text{ above it},
  $$
  with velocity $c = 20$ and $b = (1, \dots, 1)$. It is nonsymmetric, so its $\bar H_k$ is full above the subdiagonal (Step 3, Figure 3.2). This step also uses $c_k$ for the cosine of a rotation; those always carry a subscript.
- **`shift_n`** (new in this step): the $n \times n$ cyclic shift, here with $n = 8$,
  $$
  A = \begin{pmatrix}
  0 &   &        &   & 1 \\
  1 & 0 &        &   &   \\
    & 1 & \ddots &   &   \\
    &   & \ddots & 0 &   \\
    &   &        & 1 & 0
  \end{pmatrix}, \qquad b = e_1 = (1, 0, \dots, 0) ,
  $$
  with ones just below the diagonal and in the top-right corner, and zeros elsewhere. It moves each coordinate one place down and wraps the last one around to the top: $A e_j = e_{j+1}$ and $A e_n = e_1$, where $e_j$ is the $j$-th unit vector of $\mathbb{R}^n$. $A$ is a permutation matrix, so it is orthogonal and its condition number is 1, about as well behaved as a matrix gets. Section 4.7 uses it for a residual that doesn't move at all for $n - 1$ steps. Step 5 explains why.

`%autoreload 2` picks up edits to `tutorial_utils.py` without restarting the kernel.

In [ ]:
%load_ext autoreload
%autoreload 2

import time

import numpy as np  # noqa: I001
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

from tutorial_utils import (
    # the matvec wrapper from Step 1
    CountingMatvec,
    # loading definitions from an earlier step's notebook
    import_from_notebook,
    # example problems; each returns (A, b)
    geo3, diag_narrow, diag_spread, convdiff_n, shift_n,
    # checking one solver version against another
    compare_versions,
    # plot style
    style_axes, plot_residuals, plot_matrix, add_figure_caption, REFERENCE_COLOR, SERIES_COLORS,
)

np.set_printoptions(precision=4, suppress=True)

A_geo3, b_geo3 = geo3()              # 3 x 3, the running example from Steps 1-3
A_narrow, b_narrow = diag_narrow()   # 100 x 100 diagonal, eigenvalues in [1, 1.5]
A_spread, b_spread = diag_spread()   # 20 x 20 diagonal, eigenvalues from 1 to 1e4
A_cd, b_cd = convdiff_n(20, c=20.0)  # 20 x 20 tridiagonal, nonsymmetric
A_shift, b_shift = shift_n(8)        # 8 x 8 cyclic shift, b = e_1

**From Steps 2 and 3.** From Step 3 we need `gmres_v2`, the solver this step improves. From Step 2 we need the Arnoldi step `arnoldi_step_classical_gram_schmidt_twice`, which fills $v_{k+1}$ and returns column $k$ of $\bar H$, and `arnoldi`, which runs $k$ steps and returns $V_{k+1}$, $\bar H_k$ and $\beta$. Instead of copying them, `import_from_notebook` loads them straight from [GMRES_3_arnoldi_least_squares.ipynb](GMRES_3_arnoldi_least_squares.ipynb) and [GMRES_2_arnoldi_orthogonalise.ipynb](GMRES_2_arnoldi_orthogonalise.ipynb). It runs only those notebooks' imports and function definitions, not their plots or examples. To read any of them here, run `gmres_v2??` and so on.

In [ ]:
# Built in Steps 2 and 3. Loaded from those notebooks, so each stays the only copy.
gmres_v2 = import_from_notebook("GMRES_3_arnoldi_least_squares.ipynb", "gmres_v2")
arnoldi_step_classical_gram_schmidt_twice, arnoldi = import_from_notebook(
    "GMRES_2_arnoldi_orthogonalise.ipynb", "arnoldi_step_classical_gram_schmidt_twice", "arnoldi")

### 4.1 Rotating the small problem into a triangular one

A square matrix $\Omega$ is **orthogonal** if $\Omega^\mathsf{T} \Omega = I$. Like $V_{k+1}$ in Step 3, it preserves lengths: $\|\Omega z\|_2^2 = z^\mathsf{T} \Omega^\mathsf{T} \Omega\, z = \|z\|_2^2$ for every vector $z$. So multiplying the residual of the small problem by any $(k+1) \times (k+1)$ orthogonal matrix changes nothing:

$$
\big\| \beta e_1 - \bar H_k\, y \big\|_2 = \big\| \Omega\, \beta e_1 - \Omega \bar H_k\, y \big\|_2
\qquad \text{for every } y .
$$

Same minimiser $y$, same minimum. Now suppose we can choose $\Omega$ so that $\Omega \bar H_k$ is upper triangular. A $(k+1) \times k$ matrix that is upper triangular has a last row of zeros, so

$$
\Omega_k \bar H_k = \bar R_k = \begin{pmatrix} R_k \\ 0 \;\cdots\; 0 \end{pmatrix},
\qquad
\Omega_k\, \beta e_1 = g = \begin{pmatrix} g_1 \\ \vdots \\ g_k \\ g_{k+1} \end{pmatrix},
$$

where:

- $\Omega_k$ is that orthogonal $(k+1) \times (k+1)$ matrix (the subscript says it is built for $\bar H_k$),
- $\bar R_k$ is the $(k+1) \times k$ upper-triangular result, and $R_k$ its top $k \times k$ block,
- $g \in \mathbb{R}^{k+1}$ is the rotated right-hand side, with entries $g_1, \dots, g_{k+1}$.

(Written as $\bar H_k = \Omega_k^\mathsf{T} \bar R_k$, an orthogonal matrix times an upper-triangular one, this is a **QR factorisation** of $\bar H_k$.)

The rotated residual splits into its first $k$ entries and its last one:

$$
\big\| g - \bar R_k\, y \big\|_2^2 = \big\| g_{1:k} - R_k\, y \big\|_2^2 + g_{k+1}^2 ,
$$

where $g_{1:k} = (g_1, \dots, g_k)$. The last entry of $\bar R_k\, y$ is zero whatever $y$ is, so no choice of $y$ touches $g_{k+1}$. The first term can be made exactly zero by solving the triangular system $R_k\, y = g_{1:k}$. ($R_k$ is invertible as long as no $h_{j+1,j}$ is zero, since then the columns of $\bar H_k$ are independent. Step 6 deals with the case where one is.) Therefore

$$
\|r_k\|_2 = \min_{y} \big\| \beta e_1 - \bar H_k\, y \big\|_2 = |g_{k+1}| .
$$

**The residual norm is the last entry of the rotated right-hand side**, and finding it doesn't need $y$. Check it on `geo3` at $k = 2$, taking $\Omega_2$ from NumPy's QR factorisation for now. Step 1 found $\|r_2\|_2 = 0.0933$.

In [ ]:
_, H_bar_2, beta = arnoldi(CountingMatvec(A_geo3), b_geo3, 2)   # H-bar_2 is 3 x 2
Q, R_bar_2 = np.linalg.qr(H_bar_2, mode="complete")             # H-bar_2 = Q R-bar_2, Q orthogonal 3 x 3
Omega_2 = Q.T                                                    # so Omega_2 H-bar_2 = R-bar_2
g = Omega_2 @ np.array([beta, 0.0, 0.0])

print("R-bar_2 = Omega_2 H-bar_2 =\n", R_bar_2)
print("g = Omega_2 beta e_1 =", g)
print(f"|g_3| = {abs(g[2]):.4f}    (Step 1: ||r_2|| = 0.0933)")

The last row of $\bar R_2$ is zero and $|g_3| = 0.0933$, as promised. (NumPy's factor happens to have some rows with flipped signs compared with what we'll build below. A sign doesn't change a length, so it doesn't matter.) Solving a QR factorisation from scratch at every step would cost as much as `lstsq`, though. The rest of this step builds $\Omega_k$ out of small pieces that can be reused from one step to the next.

### 4.2 One Givens rotation

The pieces are rotations of a plane, the simplest orthogonal matrices that can make an entry zero. Take two numbers $(p, q)$, not both zero, think of them as a point in the plane, and let $\rho$ be its distance from the origin and $\theta$ its angle from the horizontal axis, so that $p = \rho \cos\theta$ and $q = \rho \sin\theta$. Rotating the point by $-\theta$ puts it on the horizontal axis:

$$
\begin{pmatrix} c & s \\ -s & c \end{pmatrix}
\begin{pmatrix} p \\ q \end{pmatrix}
=
\begin{pmatrix} \rho \\ 0 \end{pmatrix},
\qquad
\rho = \sqrt{p^2 + q^2}, \quad
c = \frac{p}{\rho} = \cos\theta, \quad
s = \frac{q}{\rho} = \sin\theta .
$$

Multiplying out: $c\,p + s\,q = (p^2 + q^2)/\rho = \rho$ and $-s\,p + c\,q = (-qp + pq)/\rho = 0$. Since $c^2 + s^2 = 1$, the matrix is orthogonal. If $p = q = 0$ there is nothing to rotate, and we take $c = 1$, $s = 0$, the identity. This 2×2 matrix is a **Givens rotation**, and $c$ and $s$ are its cosine and sine.

`givens_rotation(top, bottom)` computes $\rho$, $c$ and $s$ for the pair $(p, q) = $ `(top, bottom)`. It gets $\rho$ from `np.hypot`, which computes $\sqrt{p^2 + q^2}$ without overflowing when $p$ or $q$ is huge. Figure 4.1 shows it on $(p, q) = (3, 2)$, where $\rho = \sqrt{13}$.

In [ ]:
def givens_rotation(top, bottom):
    """The rotation that takes the pair (top, bottom) to (radius, 0).

    Returns:
        radius: sqrt(top^2 + bottom^2), the length of the pair.
        cosine, sine: top / radius and bottom / radius, so that
            [[cosine, sine], [-sine, cosine]] @ (top, bottom) = (radius, 0).
            The identity, (1, 0), when top and bottom are both zero.
    """
    radius = np.hypot(top, bottom)
    if radius == 0.0:
        return 0.0, 1.0, 0.0
    return radius, top / radius, bottom / radius


p, q = 3.0, 2.0
radius, cosine, sine = givens_rotation(p, q)
rotated = np.array([[cosine, sine], [-sine, cosine]]) @ np.array([p, q])
print(f"rho = {radius:.4f}  (sqrt(13) = {np.sqrt(13):.4f})    c = {cosine:.4f}    s = {sine:.4f}")
print("rotated pair:", rotated)

theta = np.arctan2(q, p)
fig, ax = plt.subplots(figsize=(7, 3.8), constrained_layout=True)
ax.annotate("", xy=(p, q), xytext=(0, 0),
            arrowprops=dict(arrowstyle="-|>", color=SERIES_COLORS[0], linewidth=2, mutation_scale=15))
ax.annotate("", xy=rotated, xytext=(0, 0),
            arrowprops=dict(arrowstyle="-|>", color=SERIES_COLORS[1], linewidth=2, mutation_scale=15))
swept = np.linspace(theta, 0, 60)      # the path of the tip during the rotation
ax.plot(radius * np.cos(swept), radius * np.sin(swept), color=REFERENCE_COLOR, linewidth=1, linestyle="--")
angle = np.linspace(0, theta, 40)
ax.plot(0.9 * np.cos(angle), 0.9 * np.sin(angle), color=REFERENCE_COLOR, linewidth=1)
ax.text(1.05 * np.cos(theta / 2), 1.05 * np.sin(theta / 2), r"$\theta$", color=REFERENCE_COLOR, fontsize=12,
        va="center")
ax.plot([p, p], [0, q], color=REFERENCE_COLOR, linewidth=1, linestyle=":")
ax.text(p + 0.08, q, r"$(p, q) = (\rho \cos\theta,\ \rho \sin\theta)$", color=SERIES_COLORS[0], fontsize=10,
        va="bottom")
ax.text(radius, -0.12, r"$(\rho, 0)$", color=SERIES_COLORS[1], fontsize=10, ha="center", va="top")
ax.set_xlim(-0.3, 5.6)
ax.set_ylim(-0.6, 2.6)
ax.set_aspect("equal")
ax.set_title(r"A Givens rotation by $-\theta$")
style_axes(ax)
add_figure_caption(fig, "4.1", r"A Givens rotation turns $(p, q) = (3, 2)$ into $(\rho, 0)$, keeping its length.")
plt.show()

In a bigger matrix, a Givens rotation acts on two neighbouring entries and leaves all the others alone. $G_j$ denotes the rotation of entries $j$ and $j+1$, with cosine $c_j$ and sine $s_j$: the identity, except for the 2×2 block in rows and columns $j$ and $j + 1$,

$$
G_j =
\begin{pmatrix}
I &                &              &   \\
  & c_j            & s_j          &   \\
  & -s_j           & c_j          &   \\
  &                &              & I
\end{pmatrix}
\begin{matrix} \\ \leftarrow \text{row } j \\ \leftarrow \text{row } j+1 \\ \\ \end{matrix}
$$

Multiplying a matrix by $G_j$ on the left replaces rows $j$ and $j+1$ by combinations of themselves and leaves every other row as it was. $G_j$ is orthogonal, and so is any product of them, since $(G G')^\mathsf{T} (G G') = G'^\mathsf{T} G^\mathsf{T} G\, G' = I$. `givens_matrix` writes $G_j$ out as a full matrix, which is useful for looking at, though no solver will ever form one. Like the arrays in Step 2, it counts rows from 0, so $G_j$ is `givens_matrix(size, j - 1, ...)`.

In [ ]:
def givens_matrix(size, row, cosine, sine):
    """The size x size identity with a Givens rotation in rows and columns row, row + 1.

    For looking at only: the solvers apply rotations to two entries at a time
    and never form this matrix. In the notes' 1-based notation it is G_{row+1}.
    """
    G = np.eye(size)
    G[row : row + 2, row : row + 2] = [[cosine, sine], [-sine, cosine]]
    return G

### 4.3 Triangularising $\bar H_k$: one rotation per column

An upper Hessenberg matrix has only one nonzero below the diagonal in each column, $h_{j+1,j}$. So it can be made upper triangular with $k$ rotations, one per column:

- $G_1$ uses rows 1 and 2 to zero the entry in position $(2, 1)$,
- $G_2$ uses rows 2 and 3 to zero the entry in position $(3, 2)$,
- and so on, up to $G_k$, which zeroes position $(k+1, k)$.

Then $\Omega_k = G_k \cdots G_2\, G_1$. Each $G_j$ is built from the two entries in column $j$ that it rotates, *as they are after* $G_1, \dots, G_{j-1}$: the diagonal entry at position $(j, j)$ plays $p$ and the subdiagonal entry $h_{j+1,j}$ plays $q$. A later rotation never fills in a zero that an earlier one made, because when $G_j$ acts, rows $j$ and $j + 1$ are both already zero to the left of column $j$ (row $j+1$ because $\bar H_k$ is Hessenberg, and row $j$ because $G_{j-1}$ zeroed its last entry there). So the zeros made earlier stay zero.

(A full $(k+1) \times k$ matrix would need one rotation for every entry below its diagonal, about $k^2/2$ of them. The Hessenberg shape from Step 3 is what makes this cheap.)

By hand on `geo3`, with $\bar H_3$, which is $4 \times 3$. Its entry $h_{43}$ is exactly 0: at $k = 3 = n$ the Krylov space is all of $\mathbb{R}^3$, and Step 2's Arnoldi step flagged the breakdown (section 2.4). The rotations go to $\beta e_1$ as well, to get $g$.

In [ ]:
_, H_bar_3, beta = arnoldi(CountingMatvec(A_geo3), b_geo3, 3)
print("H-bar_3 =\n", H_bar_3)

R_bar_geo3 = H_bar_3.copy()              # becomes Omega_3 H-bar_3 = R-bar_3
g_geo3 = np.array([beta, 0.0, 0.0, 0.0])  # becomes Omega_3 beta e_1
for j in range(1, 4):                     # G_1, G_2, G_3, in the notes' 1-based numbering
    radius, cosine, sine = givens_rotation(R_bar_geo3[j - 1, j - 1], R_bar_geo3[j, j - 1])
    G = givens_matrix(4, j - 1, cosine, sine)
    R_bar_geo3 = G @ R_bar_geo3
    g_geo3 = G @ g_geo3
    print(f"\nafter G_{j}:  c_{j} = {cosine:.4f},  s_{j} = {sine:.4f}")
    print(R_bar_geo3)

print("\ng = Omega_3 beta e_1 =", g_geo3)

Each rotation changes only two rows, and each one leaves a zero where the subdiagonal entry was, so after three rotations $\bar R_3$ is upper triangular with diagonal $3.3665, 2.2524, 1.7144$. The first rotation turns the pair $(h_{11}, h_{21}) = (10/3, \sqrt 2/3)$ by only $\theta = 8^\circ$ ($s_1 = 0.1400$). The last one is exactly the identity, $c_3 = 1$ and $s_3 = 0$, since $h_{43} = 0$ leaves nothing below the diagonal to remove. So $g_4 = -s_3\, g_3$ is exactly zero: at $k = 3$ the residual is zero, as Step 1 found.

Figure 4.2 does the same on `convdiff_n` with $k = 6$, one rotation per panel. The outlined rows in each panel are the two rows that rotation changed. Entries are colored by size on a log scale, and blank cells are exact zeros.

In [ ]:
_, H_cd_6, _ = arnoldi(CountingMatvec(A_cd), b_cd, 6)   # H-bar_6, 7 x 6

# One color scale for all panels.
smallest = np.abs(H_cd_6[H_cd_6 != 0]).min() / 10
largest = np.abs(H_cd_6).max() * 1.5

fig, axes = plt.subplots(1, 7, figsize=(17, 3.4), constrained_layout=True)
R_bar = H_cd_6.copy()
for j, ax in enumerate(axes):   # panel j: after G_1, ..., G_j
    if j > 0:
        radius, cosine, sine = givens_rotation(R_bar[j - 1, j - 1], R_bar[j, j - 1])
        R_bar = givens_matrix(7, j - 1, cosine, sine) @ R_bar
        R_bar[j, j - 1] = 0.0   # zero in exact arithmetic; store it as exactly zero
        ax.add_patch(Rectangle((0.5, j - 0.5), 6, 2, fill=False, edgecolor=SERIES_COLORS[1], linewidth=2))
    title = r"$\bar H_6$" if j == 0 else rf"after $G_{{{j}}}$"
    plot_matrix(ax, R_bar, log=True, vmin=smallest, vmax=largest, title=title, colorbar=(j == 6))
    if j > 0:
        ax.set_ylabel("")
add_figure_caption(fig, "4.2", r"Six Givens rotations turn $\bar H_6$ of convdiff_n (c = 20) into $\bar R_6$. Rotation $G_j$ changes only rows $j$ and $j+1$ (outlined) and zeroes the entry below the diagonal in column $j$.")
plt.show()

The subdiagonal disappears one entry at a time, from the top left. Look at which entries each rotation leaves alone. $G_j$ only changes rows $j$ and $j+1$, and in those rows only columns $j$ onwards, so once $G_j$ has acted, **column $j$ of $\bar R$ is final**: no later rotation touches its entries 1 to $j$. The same holds for $g$: after $G_j$, its entries $g_1, \dots, g_j$ are final. That is what makes the next section work.

### 4.4 One column at a time

In GMRES, $\bar H_k$ doesn't arrive all at once. Step $k$ of Arnoldi produces column $k$, with entries $h_{1k}, \dots, h_{k+1,k}$. The rotations $G_1, \dots, G_{k-1}$ were built from columns $1, \dots, k-1$ only, so they are already known, and they don't change when column $k$ arrives. That gives the update for step $k$:

1. **Apply the old rotations to the new column**, $G_1$ first, then $G_2$, up to $G_{k-1}$. Call its entry $k$ after that $\tilde h_{kk}$. (Its last entry, $h_{k+1,k}$, isn't changed: those rotations only touch entries 1 to $k$.)
2. **Build one new rotation** $G_k$ from the pair $(\tilde h_{kk},\ h_{k+1,k})$:
   $$
   R_{kk} = \sqrt{\tilde h_{kk}^2 + h_{k+1,k}^2}, \qquad
   c_k = \frac{\tilde h_{kk}}{R_{kk}}, \qquad
   s_k = \frac{h_{k+1,k}}{R_{kk}} .
   $$
   Column $k$ of $R_k$ is the rotated entries 1 to $k-1$, with $R_{kk}$ on the diagonal.
3. **Rotate the right-hand side.** Before this step, $g = (g_1, \dots, g_k, 0)$, and $|g_k| = \|r_{k-1}\|_2$ was the last residual. $G_k$ changes only entries $k$ and $k + 1$:
   $$
   g_k \leftarrow c_k\, g_k , \qquad g_{k+1} = -s_k\, g_k .
   $$

So the new residual is

$$
\boxed{\ \|r_k\|_2 = |g_{k+1}| = |s_k|\, \|r_{k-1}\|_2\ }
$$

**$|s_k|$ is the factor by which step $k$ shrinks the residual.** Three things follow:

- **The residual never increases**, because $|s_k| \le 1$ (from $c_k^2 + s_k^2 = 1$). Step 1 argued this from the nested Krylov spaces. Here it is one line of algebra.
- **A step makes no progress exactly when $|s_k| = 1$**, which means $c_k = 0$, that is $\tilde h_{kk} = 0$. Section 4.7 shows a matrix where that happens.
- **The residual drops to zero exactly when $s_k = 0$**, which means $h_{k+1,k} = 0$: Arnoldi found no new direction, and the answer is exact. That is a breakdown, and Step 6 is about it.

The same thing on `geo3`, column by column. Each rotation is applied to two entries at a time, without forming $G_j$. The residuals should be Step 1's $1.732 \to 0.2425 \to 0.0933 \to 0$.

In [ ]:
_, H_bar_3, beta = arnoldi(CountingMatvec(A_geo3), b_geo3, 3)
R = np.zeros((3, 3))                 # R_3, filled one column per step
g = np.array([beta, 0.0, 0.0, 0.0])  # g, rotated one step at a time
rotations = []                       # (c_j, s_j) for G_1, G_2, ...

print(f"k = 0:  ||r_0|| = |g_1| = {abs(g[0]):.4f}")
for k in range(1, 4):
    column = H_bar_3[:, k - 1].copy()  # column k of H-bar, as Arnoldi step k returns it

    # 1. The old rotations, in order, each on two entries of the column.
    for i, (c_i, s_i) in enumerate(rotations):
        column[i], column[i + 1] = c_i * column[i] + s_i * column[i + 1], -s_i * column[i] + c_i * column[i + 1]

    # 2. One new rotation from (h-tilde_kk, h_{k+1,k}).
    radius, cosine, sine = givens_rotation(column[k - 1], column[k])
    rotations.append((cosine, sine))
    R[: k - 1, k - 1] = column[: k - 1]
    R[k - 1, k - 1] = radius

    # 3. Rotate g: only entries k and k + 1 change.
    g[k - 1], g[k] = cosine * g[k - 1], -sine * g[k - 1]
    print(f"k = {k}:  s_{k} = {sine:.4f}    ||r_{k}|| = |g_{k + 1}| = {abs(g[k]):.4f}")

print("\nR_3 =\n", R)
print("g =", g)
print("same R and g as rotating the whole matrix in 4.3:",
      np.allclose(R, R_bar_geo3[:3]) and np.allclose(g, g_geo3))

The residuals are Step 1's, $1.732 \to 0.2425 \to 0.0933 \to 0$, and $R_3$ and $g$ are the ones 4.3 got by rotating the whole matrix. Each residual is the previous one times $|s_k|$: $1.732 \times 0.1400 = 0.2425$ and $0.2425 \times 0.3845 = 0.0933$.

### 4.5 `gmres_v3`

**The algorithm, `gmres_v3`:**

1. Allocate `V` ($n \times (m+1)$), `R` ($m \times m$) and `g` (length $m + 1$) as zeros, with $m$ = `max_iterations`. Set $v_1 = b / \beta$, $g_1 = \beta$, and start an empty list of rotations.
2. For $k = 1, 2, \dots$:
   1. One Arnoldi step: it fills $v_{k+1}$ and returns column $k$ of $\bar H$. *(1 matvec.)*
   2. Apply the stored rotations $G_1, \dots, G_{k-1}$ to that column.
   3. Build $G_k$, store $(c_k, s_k)$, and write column $k$ of $R$.
   4. Rotate $g$: $g_k \leftarrow c_k g_k$, $g_{k+1} = -s_k g_k$. Now $\|r_k\|_2 = |g_{k+1}|$.
   5. Stop if $\|r_k\|_2 \le \max(\text{rtol} \cdot \|b\|_2,\ \text{atol})$, or after a breakdown.
3. Solve $R_k\, y = g_{1:k}$ by back-substitution and form $x_k = V_k\, y$, once.

Compared with `gmres_v2`, the small problem never gets solved inside the loop. Step $k$ spends $k - 1$ rotations of two numbers each on it, instead of a least-squares solve of a $(k+1) \times k$ matrix, and $\bar H$ isn't even stored: each column goes into $R$ as soon as it arrives. Section 4.8 measures the difference.

**Back-substitution.** $R_k$ is upper triangular, so its last equation, $R_{kk}\, y_k = g_k$, has one unknown. Solve it, substitute $y_k$ into the equation above it, which then has one unknown too, and so on up:

$$
y_i = \frac{g_i - \sum_{j > i} R_{ij}\, y_j}{R_{ii}}, \qquad i = k, k-1, \dots, 1 .
$$

**A new keyword, `atol`.** The stopping test becomes $\|r_k\|_2 \le \max(\text{rtol} \cdot \|b\|_2,\ \text{atol})$, an absolute floor under the relative tolerance, as in production. The default, `atol=0`, gives `gmres_v2`'s test. A floor is for a caller that knows what residual size is good enough whatever $\|b\|_2$ is: when $\|b\|_2$ is small, $\text{rtol} \cdot \|b\|_2$ alone can ask for far more than it needs. (In the simulation, $b$ is the residual of a Newton iteration, which shrinks as Newton converges.)

In [ ]:
def back_substitution(R, z):
    """Solve R y = z for an upper-triangular R, last unknown first."""
    k = z.shape[0]
    y = np.zeros(k)
    for i in reversed(range(k)):
        y[i] = (z[i] - R[i, i + 1 :] @ y[i + 1 :]) / R[i, i]
    return y


def gmres_v3(matvec, b, *, rtol=1e-10, atol=0.0, max_iterations=None, verbose=False):
    """GMRES with a Givens QR of H-bar, updated one column per step.

    The residual norm of each step is read off the rotated right-hand side,
    |g_{k+1}|, without solving the small problem; x_k is formed only once, at
    the end, by back-substitution. Same iterates as gmres_v2.

    Args:
        matvec: function v -> A v. The solver never sees A itself.
        b: right-hand side, shape (n,).
        rtol: stop once ||r_k|| <= max(rtol * ||b||, atol).
        atol: absolute floor of that tolerance.
        max_iterations: largest k to try. Defaults to n.
        verbose: print s_k and the residual at every step.

    Returns:
        x: the last iterate x_k.
        record: dict with
            "iterations"        number of steps k taken,
            "matvecs"           number of matvec calls,
            "residual_history"  [||r_0||, ||r_1||, ..., ||r_k||],
            "converged"         whether the tolerance was reached.
    """
    n = b.shape[0]
    if max_iterations is None:
        max_iterations = n
    b_norm = np.linalg.norm(b)
    beta = b_norm
    tolerance = max(rtol * b_norm, atol)

    # ----- k = 0: start from x_0 = 0, so the residual is b and v_1 = b / beta. -----
    residual_history = [b_norm]
    matvec_count = 0
    converged = False

    m = max_iterations
    V = np.zeros((n, m + 1))  # columns v_1, ..., v_{m+1}, filled one per step
    R = np.zeros((m, m))      # R_k, the triangular factor, filled one column per step
    g = np.zeros(m + 1)       # Omega_k beta e_1, the rotated right-hand side
    g[0] = beta
    rotations = []            # (c_j, s_j) of G_1, ..., G_k
    V[:, 0] = b / beta

    for k in range(1, max_iterations + 1):
        # 1. One Arnoldi step: v_{k+1} and column k of H-bar. Its only matvec.
        h, breakdown = arnoldi_step_classical_gram_schmidt_twice(matvec, V, k - 1)
        matvec_count += 1

        # 2. The earlier rotations G_1, ..., G_{k-1}, in order, on the new column.
        for i, (c_i, s_i) in enumerate(rotations):
            h[i], h[i + 1] = c_i * h[i] + s_i * h[i + 1], -s_i * h[i] + c_i * h[i + 1]

        # 3. One new rotation G_k zeroes h_{k+1,k}; column k of R is then final.
        radius, cosine, sine = givens_rotation(h[k - 1], h[k])
        rotations.append((cosine, sine))
        R[: k - 1, k - 1] = h[: k - 1]
        R[k - 1, k - 1] = radius

        # 4. G_k on the right-hand side. Its new last entry is the residual norm.
        g[k - 1], g[k] = cosine * g[k - 1], -sine * g[k - 1]
        residual_history.append(abs(g[k]))

        if verbose:
            print(f"k = {k}:  |s_k| = {abs(sine):.4f},  ||r_k|| = {residual_history[-1]:.4e}")

        # 5. Stop once the residual is small enough.
        if residual_history[-1] <= tolerance:
            converged = True
            break
        # 6. No new direction: the Krylov space has stopped growing (Step 6).
        if breakdown:
            break

    # The iterate is needed only now: R_k y = g_{1:k}, then x_k = V_k y.
    y = back_substitution(R[:k, :k], g[:k])
    x = V[:, :k] @ y

    record = {
        "iterations": k,
        "matvecs": matvec_count,
        "residual_history": residual_history,
        "converged": converged,
    }
    return x, record

In [ ]:
x, record = gmres_v3(CountingMatvec(A_geo3), b_geo3, verbose=True)
print("x =", x, "   matvecs:", record["matvecs"])
print()

for name, A, b, ks in [("geo3", A_geo3, b_geo3, None),
                       ("diag_narrow", A_narrow, b_narrow, range(1, 12)),
                       ("diag_spread", A_spread, b_spread, None),
                       ("convdiff_n", A_cd, b_cd, None)]:
    diff = compare_versions(gmres_v2, gmres_v3, A, b, ks=ks)
    print(f"{name:12s} gmres_v2 vs gmres_v3, k = 1..{diff['k'][-1]}:  "
          f"largest x_difference {diff['x_difference'].max():.1e},  "
          f"largest residual_difference {diff['residual_difference'].max():.1e}")
    assert diff["matvecs_equal"].all()
    assert diff["x_difference"].max() < 1e-11 and diff["residual_difference"].max() < 1e-11

The verbose run on `geo3` prints the $|s_k|$ and residuals of 4.4, and the exact solution $x = (3, 4, 5)/13$ after 3 matvecs. On all four matrices `gmres_v2` and `gmres_v3` agree to round-off, with the same matvec counts. The largest difference between iterates is $8.9 \times 10^{-14}$, on `diag_spread`; on the other three it is about $10^{-14}$ or less.

### 4.6 How far down can the residual go?

`gmres_v3` never computes the residual vector $b - A x_k$. Its residual is $|g_{k+1}|$, which equals $\|r_k\|_2$ in exact arithmetic for two reasons: the small problem has the same residual as the big one (Step 3, which needs $V_{k+1}$ orthonormal), and the rotations don't change the small problem's residual (4.1). Figure 4.3 checks this in floating point. It runs both solvers with `rtol=0` up to $k = 20$, past where they would normally stop, and compares three numbers at each step:

- `gmres_v3`'s $|g_{k+1}|$,
- `gmres_v2`'s small-problem residual $\|\beta e_1 - \bar H_k\, y\|_2$, which `lstsq` computes from its $y$,
- the true residual $\|b - A x_k\|_2$ of `gmres_v3`'s iterate, which `compare_versions` measures using $A$ directly.

The left panel is `diag_spread`, which needs all 20 steps. The right panel is `diag_narrow`, which reaches $10^{-10}$ at $k = 11$ and then keeps going. An exact zero has no place on a log axis, so one is drawn on the bottom edge of its panel, as in Figure 4.4 below.

In [ ]:
bottom = 1e-21   # where an exact zero is drawn
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4), sharey=True, constrained_layout=True)
for ax, (name, A, b) in zip(axes, [("diag_spread", A_spread, b_spread), ("diag_narrow", A_narrow, b_narrow)]):
    b_norm = np.linalg.norm(b)
    _, record_v2 = gmres_v2(CountingMatvec(A), b, rtol=0.0, max_iterations=20)
    _, record_v3 = gmres_v3(CountingMatvec(A), b, rtol=0.0, max_iterations=20)
    diff = compare_versions(gmres_v2, gmres_v3, A, b, ks=range(1, 21))

    plot_residuals(ax, np.maximum(record_v3["residual_history"], bottom * b_norm), b_norm=b_norm,
                   color=SERIES_COLORS[0], label=r"gmres_v3: $|g_{k+1}|$")
    plot_residuals(ax, record_v2["residual_history"], b_norm=b_norm, color=SERIES_COLORS[1],
                   marker="none", linestyle="--", label=r"gmres_v2: $\|\beta e_1 - \bar H_k y\|_2$")
    ax.semilogy(diff["k"], diff["true_residual_new"], color=SERIES_COLORS[2], linestyle="none",
                marker="x", markersize=8, markeredgewidth=2, label=r"gmres_v3's $x_k$: true $\|b - A x_k\|_2$")
    ax.set_title(f"{name} (n = {b.shape[0]}), rtol = 0")
    style_axes(ax, xticks=range(0, 21, 2))

    print(f"{name}:")
    for k in (11, 14, 16, 19, 20):
        print(f"  k = {k:2d}:  |g_k+1| = {record_v3['residual_history'][k] / b_norm:.1e}    "
              f"v2 small problem = {record_v2['residual_history'][k] / b_norm:.1e}    "
              f"true = {diff['true_residual_new'][k - 1]:.1e}")
axes[0].set_ylim(bottom / 2, 3)
axes[0].legend(frameon=False, loc="lower left")
add_figure_caption(fig, "4.3", r"gmres_v3's $|g_{k+1}|$, gmres_v2's small-problem residual and the true residual. Left: they agree until k = 20 = n, where a breakdown makes $|g_{21}|$ exactly 0 (bottom edge). Right: below about $10^{-15}$, $|g_{k+1}|$ keeps falling while the true residual stays at round-off.")
plt.show()

**`diag_spread` (left).** The three curves lie on top of each other down to $1.3 \times 10^{-2}$ at $k = 19$. At $k = 20 = n$ the Krylov space is all of $\mathbb{R}^{20}$, and the Arnoldi step is flagged as a breakdown, as Step 3 found: $h_{21,20} = 0$. Then $s_{20} = 0$, and `gmres_v3` reports $|g_{21}| = 0$ exactly (on the bottom edge). The true residual of its iterate is $1.5 \times 10^{-13}$, and `gmres_v2`'s `lstsq` residual is $6.9 \times 10^{-13}$. At a breakdown the small problem can be solved exactly, so its residual says 0, but the iterate built from it still carries round-off. That is one reason production checks the true residual after a breakdown (Step 6).

**`diag_narrow` (right).** The three agree down to about $10^{-15}$ at $k = 15$, and then they part. The true residual stops at about $1.6 \times 10^{-16}$, and `gmres_v2`'s small-problem residual at about $10^{-15}$ (it is a difference of nearly equal numbers, so it carries its own round-off). The residual has reached **round-off**. Computing $A x_k$ in float64 makes relative errors of about $\varepsilon_{\text{machine}} \approx 2.2 \times 10^{-16}$, so the true residual of any computed iterate is around $\varepsilon_{\text{machine}} \|A\|_2 \|x_k\|_2$, which for this matrix is a few times $10^{-16}\, \|b\|_2$.

But $|g_{k+1}|$ keeps falling, by $|s_k| \approx 0.1$ a step: $1.6 \times 10^{-16}$ at $k = 16$, $1.6 \times 10^{-20}$ at $k = 20$. It knows nothing about the round-off in $A x_k$. By 4.4 it is a product of sines, $|g_{k+1}| = \beta\, |s_1 s_2 \cdots s_k|$, and each new rotation shrinks it whether or not the true residual can follow.

So below about $10^{-15}$, $|g_{k+1}|$ reports progress the true residual doesn't make, and a tolerance set there is met on paper only. On `diag_narrow`:

In [ ]:
for rtol in (1e-10, 1e-14, 1e-15, 1e-16):
    x, record = gmres_v3(CountingMatvec(A_narrow), b_narrow, rtol=rtol)
    print(f"rtol = {rtol:.0e}:  {record['iterations']:2d} steps    "
          f"|g_k+1| = {record['residual_history'][-1] / np.linalg.norm(b_narrow):.1e}    "
          f"true = {np.linalg.norm(b_narrow - A_narrow @ x) / np.linalg.norm(b_narrow):.1e}")

With `rtol=1e-16` the solver stops after 17 steps, with $|g_{k+1}| = 1.6 \times 10^{-17}$, and reports that it converged. The true residual of the iterate it returns is $1.6 \times 10^{-16}$: ten times what $|g_{k+1}|$ claims, above the tolerance the solver says it met, and hardly better than the $2.1 \times 10^{-16}$ it had at $k = 16$. A tolerance should stay well above $\varepsilon_{\text{machine}}$.

$|g_{k+1}|$ can part from the true residual in a second way, which Step 3's demonstration showed: when the basis $V_{k+1}$ isn't orthonormal, the small problem stops describing the big one, and $|g_{k+1}|$ with it. Classical Gram–Schmidt applied twice rules that out here. Neither problem shows in $|g_{k+1}|$ itself, which is why production can recompute the true residual: after every breakdown (Step 6), and at the end of a solve when called with `recompute_residual=True`, which then judges convergence by the true residual (Step 9).

### 4.7 What $|s_k|$ says

By 4.4, $|s_k| = \|r_k\|_2 / \|r_{k-1}\|_2$, the factor by which step $k$ shrinks the residual. So the record already holds it: it is the ratio of successive entries of `residual_history`. Figure 4.4 shows the residual histories (left) and $|s_k|$ (right) for three matrices, each run with the default `rtol` of $10^{-10}$: `diag_narrow`, the nonsymmetric `convdiff_n` ($c = 20$) and the cyclic shift `shift_n` ($n = 8$).

In [ ]:
fig, (ax_residual, ax_sine) = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=True)
for color, (name, A, b) in zip(SERIES_COLORS, [("diag_narrow", A_narrow, b_narrow),
                                               ("convdiff_n, c = 20", A_cd, b_cd),
                                               ("shift_n, n = 8", A_shift, b_shift)]):
    _, record = gmres_v3(CountingMatvec(A), b)
    residuals = np.array(record["residual_history"])
    sines = residuals[1:] / residuals[:-1]   # |s_k| = ||r_k|| / ||r_{k-1}||
    # An exact zero has no place on a log axis; draw it on the bottom edge of the plot.
    shown = np.maximum(residuals, 1e-17 * np.linalg.norm(b))
    plot_residuals(ax_residual, shown, b_norm=np.linalg.norm(b), color=color, label=name)
    ax_sine.plot(np.arange(1, len(sines) + 1), sines, color=color, linewidth=2, marker="o", markersize=5,
                 label=name)
    print(f"{name:20s} steps {record['iterations']:2d}    |s_k| =", np.round(sines, 4))

ax_residual.set_ylim(1e-17, 3)
ax_residual.legend(frameon=False, loc="lower left")
style_axes(ax_residual, xticks=range(0, 21, 2))
ax_sine.axhline(1.0, color=REFERENCE_COLOR, linewidth=1)
ax_sine.text(20, 1.02, r"$|s_k| = 1$: no progress", color=REFERENCE_COLOR, fontsize=9, ha="right", va="bottom")
ax_sine.set_ylim(-0.03, 1.12)
ax_sine.set_xlabel("step k")
ax_sine.set_ylabel(r"$|s_k| = \|r_k\|_2 \,/\, \|r_{k-1}\|_2$")
ax_sine.legend(frameon=False, loc="center left")
style_axes(ax_sine, xticks=range(0, 21, 2))
add_figure_caption(fig, "4.4", r"Residual histories (left) and the sines $|s_k|$ (right). Step $k$ shrinks the residual by the factor $|s_k|$. The residuals of shift_n at k = 8 and of convdiff_n at k = 20 are exactly 0, drawn on the bottom edge.")
plt.show()

The right panel reads like a summary of each problem:

- **`diag_narrow`:** $|s_k|$ is between 0.10 and 0.12 at every step, so every step gains about one digit and the residual is a straight line on the log plot, reaching $10^{-10}$ at $k = 11$.
- **`convdiff_n`:** $|s_k|$ is about 0.96 for the first ten steps. Each step removes only 4% of the residual, which is still $0.67\, \|b\|_2$ after ten steps. It speeds up only near the end, and at $k = 20 = n$, where the Krylov space is all of $\mathbb{R}^{20}$, the residual is exactly zero: the Arnoldi step flagged the breakdown, so $s_{20} = 0$ (Step 6). This is close to the worst case.
- **`shift_n`:** $|s_k| = 1$ exactly for $k = 1, \dots, 7$. The residual stays at exactly $\|b\|_2 = 1$: seven matvecs with no progress at all. At $k = 8 = n$, $s_8 = 0$ and the residual is exactly zero.

`shift_n`'s stall can be seen directly. With $b = e_1$ the Krylov space is $\mathcal{K}_k = \operatorname{span}\{e_1, \dots, e_k\}$, and $A$ moves that to $\operatorname{span}\{e_2, \dots, e_{k+1}\}$, which is perpendicular to $b$ as long as $k < n$. So for every $x \in \mathcal{K}_k$, $\|b - Ax\|_2^2 = \|b\|_2^2 + \|Ax\|_2^2 \ge \|b\|_2^2$, and the best GMRES can do is $x_k = 0$. That happens for a matrix with condition number 1. Neither the size of $A$ nor its condition number decides $|s_k|$. Step 5 shows what does.

### 4.8 What it saves: re-solving against updating

Per step, the small-problem work in `gmres_v2` is a least-squares solve of a $(k+1) \times k$ matrix, which costs on the order of $k^3$ operations. In `gmres_v3` it is $k - 1$ old rotations plus one new one, each a few operations on two numbers, so on the order of $k$. The one back-substitution at the end costs on the order of $k^2$, once.

Figure 4.5 times just that part of each step, with nothing else: `lstsq` on $\bar H_k$ against the Givens update of column $k$. To get many steps, it takes $\bar H_{200}$ from 200 Arnoldi steps on a bigger `convdiff_n`, with $n = 400$ (same $c = 20$, $b = (1, \dots, 1)$). Each time is the fastest of 5 repeats, to reduce noise from the rest of the computer.

In [ ]:
A_big, b_big = convdiff_n(400, c=20.0)
_, H_big, beta_big = arnoldi(CountingMatvec(A_big), b_big, 200)
steps = np.arange(1, H_big.shape[1] + 1)

lstsq_seconds = np.full(len(steps), np.inf)
givens_seconds = np.full(len(steps), np.inf)
for repeat in range(5):
    rotations = []
    g = np.zeros(len(steps) + 1)
    g[0] = beta_big
    for k in steps:
        beta_e1 = np.zeros(k + 1)
        beta_e1[0] = beta_big
        start = time.perf_counter()
        np.linalg.lstsq(H_big[: k + 1, :k], beta_e1, rcond=None)
        lstsq_seconds[k - 1] = min(lstsq_seconds[k - 1], time.perf_counter() - start)

        h = H_big[:, k - 1].copy()
        start = time.perf_counter()   # the Givens update, as in gmres_v3
        for i, (c_i, s_i) in enumerate(rotations):
            h[i], h[i + 1] = c_i * h[i] + s_i * h[i + 1], -s_i * h[i] + c_i * h[i + 1]
        radius, cosine, sine = givens_rotation(h[k - 1], h[k])
        rotations.append((cosine, sine))
        g[k - 1], g[k] = cosine * g[k - 1], -sine * g[k - 1]
        givens_seconds[k - 1] = min(givens_seconds[k - 1], time.perf_counter() - start)

for k in (10, 50, 100, 200):
    print(f"k = {k:3d}:  lstsq {lstsq_seconds[k - 1] * 1e3:7.3f} ms    "
          f"Givens update {givens_seconds[k - 1] * 1e3:7.3f} ms    "
          f"ratio {lstsq_seconds[k - 1] / givens_seconds[k - 1]:5.1f}")
print(f"over all {steps[-1]} steps:  lstsq {lstsq_seconds.sum():.3f} s    Givens {givens_seconds.sum():.3f} s")

fig, ax = plt.subplots(figsize=(6.5, 4), constrained_layout=True)
ax.loglog(steps, lstsq_seconds * 1e3, color=SERIES_COLORS[1], linewidth=2, label=r"gmres_v2: lstsq on $\bar H_k$")
ax.loglog(steps, givens_seconds * 1e3, color=SERIES_COLORS[0], linewidth=2, label="gmres_v3: Givens update")
ax.set_xlabel("step k")
ax.set_ylabel("time for the small problem (ms)")
ax.set_title("Small-problem work per step, convdiff_n (n = 400)")
ax.legend(frameon=False, loc="upper left")
style_axes(ax)
add_figure_caption(fig, "4.5", r"Time per step spent on the small problem: re-solving it with lstsq against updating its Givens QR by one column.")
plt.show()

At $k = 10$ the Givens update is about 3 times faster than `lstsq`, and at $k = 200$ about 35 times. Over all 200 steps the small problem takes about 0.4 s with `lstsq` and 0.02 s with Givens. (Timings, and with them the ratios, vary from run to run; how they grow with $k$ is what matters.) The Givens time grows in proportion to $k$, as expected from $k - 1$ rotations. `lstsq` grows roughly like $k^2$ in this range; its $k^3$ term takes over only for larger $k$.

Neither is large next to the Arnoldi step itself when $n$ is large: the matvec and the orthogonalisation against $k$ basis vectors cost on the order of $n k$ per step. What the Givens update buys is that the small problem never becomes the bottleneck, and that deciding whether to stop costs nothing.

### 4.9 Production's form: one matrix instead of a list

`gmres_v3` keeps the rotations as a list of $(c_j, s_j)$ pairs and applies them to a new column in a loop, one pair at a time. Production keeps their product instead, $\Omega_{k-1} = G_{k-1} \cdots G_1$, as a single $(m+1) \times (m+1)$ matrix, and applies all of them to the new column $h$ with one matrix-vector product, $\Omega_{k-1} h$. After building $G_k$, it updates the product by rotating two of its rows, $\Omega_k = G_k\, \Omega_{k-1}$. These are the same rotations applied in the same order, so they give the same numbers up to round-off.

The reason is the hardware. On a GPU, every separate operation costs a fixed launch time however small the arrays are. A loop over $k - 1$ rotations is $k - 1$ launches, while the product is one. (The [givens.py](../WhistlerRegulatedHeatDispersion/src/time_integration/implicit/givens.py) docstring measures the loop as ten times slower.) In NumPy on a CPU the loop is fine, which is why `gmres_v3` keeps the version that is easier to read.

Below, the product form on `convdiff_n` for 20 steps, checked against `gmres_v3`.

In [ ]:
m = 20
V_cd, H_cd, beta_cd = arnoldi(CountingMatvec(A_cd), b_cd, m)

Omega = np.eye(m + 1)        # Omega_{k-1} = G_{k-1} ... G_1; rows past k are still the identity
R_product = np.zeros((m, m))
g_product = np.zeros(m + 1)
g_product[0] = beta_cd
residuals_product = [beta_cd]
for k in range(1, m + 1):
    rotated = Omega @ H_cd[:, k - 1]   # all the old rotations in one product
    radius, cosine, sine = givens_rotation(rotated[k - 1], rotated[k])
    R_product[: k - 1, k - 1] = rotated[: k - 1]
    R_product[k - 1, k - 1] = radius
    # Omega_k = G_k Omega_{k-1}: rotate rows k and k + 1 (1-based) of the product, and of g.
    Omega[[k - 1, k]] = [cosine * Omega[k - 1] + sine * Omega[k], -sine * Omega[k - 1] + cosine * Omega[k]]
    g_product[k - 1], g_product[k] = cosine * g_product[k - 1], -sine * g_product[k - 1]
    residuals_product.append(abs(g_product[k]))

x_product = V_cd[:, :m] @ back_substitution(R_product, g_product[:m])
x_v3, record_v3 = gmres_v3(CountingMatvec(A_cd), b_cd, rtol=0.0, max_iterations=m)

print("Omega_20 orthogonal:  ||I - Omega^T Omega|| =", f"{np.linalg.norm(np.eye(m + 1) - Omega.T @ Omega):.1e}")
print("largest difference in the residuals:       ",
      f"{np.max(np.abs(np.array(residuals_product) - record_v3['residual_history'])) / beta_cd:.1e}")
print("difference in x_20, relative:              ",
      f"{np.linalg.norm(x_product - x_v3) / np.linalg.norm(x_v3):.1e}")

$\Omega_{20}$ is orthogonal to round-off, and the product form gives the same residuals (to $9.9 \times 10^{-17}$) and the same $x_{20}$ (to $4.3 \times 10^{-16}$) as `gmres_v3`'s list of rotations.

### Summary of Step 4

- A least-squares problem doesn't change when it is multiplied by an orthogonal matrix. If $\Omega_k \bar H_k = \bar R_k$ is upper triangular and $g = \Omega_k \beta e_1$, then $\|r_k\|_2 = |g_{k+1}|$, and $y$ solves the triangular system $R_k\, y = g_{1:k}$.
- A **Givens rotation** with $c = p/\rho$ and $s = q/\rho$, $\rho = \sqrt{p^2 + q^2}$, turns the pair $(p, q)$ into $(\rho, 0)$. Because $\bar H_k$ is Hessenberg, one rotation per column makes it triangular.
- When column $k$ arrives, it takes the old rotations $G_1, \dots, G_{k-1}$ and one new one, $G_k$. Earlier columns of $R$ and entries of $g$ never change again.
- $\|r_k\|_2 = |s_k|\, \|r_{k-1}\|_2$: the residual never increases, a step makes no progress exactly when $|s_k| = 1$, and the residual is zero exactly when $s_k = 0$ (breakdown, Step 6).
- **`gmres_v3`** reads the residual off $g$ at every step, never stores $\bar H$, and back-substitutes once at the end. Its iterates match `gmres_v2`'s to round-off, at a small-problem cost per step of order $k$ instead of $k^3$.
- $|g_{k+1}|$ tracks the true residual down to round-off, about $10^{-15}\, \|b\|_2$ on `diag_narrow`. Below that it keeps falling while the true residual doesn't, so a tolerance set there is met on paper only.
- `gmres_v3` adds the keyword `atol`, an absolute floor under the tolerance, $\max(\text{rtol} \cdot \|b\|_2,\ \text{atol})$.

**Ties to code.** [givens.py](../WhistlerRegulatedHeatDispersion/src/time_integration/implicit/givens.py) is this step:

- `GivensFactor` holds `rotation` (the product $\Omega_k$ of 4.9), `triangular` ($R$) and `rhs` ($g$).
- `start(residual_norm, restart)` sets $g = \beta e_1$, with the identity for both matrices.
- `fold(factor, column, step)` is one step of 4.4 in the product form of 4.9: `factor.rotation @ column`, then `_rotation`, which is `givens_rotation` (including the identity when both entries are zero), then `_rotate_pair` on the rotation matrix and on `rhs`.
- `residual_estimate(factor, steps)` is $|g_{k+1}|$.
- `solve(factor, steps)` is the back-substitution, through `solve_triangular`.

In [gmres.py](../WhistlerRegulatedHeatDispersion/src/time_integration/implicit/gmres.py), `gmres` sets `tolerance = max(rtol * norm(rhs), atol)`, and `_arnoldi_steps` keeps stepping while `residual_estimate` is above it. Production's triangular factor starts as the identity rather than zeros, and `solve` masks the steps not yet taken. Both come from JAX's fixed array shapes (Step 9). Production also checks each new pivot $R_{kk}$ before counting a step, which is Step 6.

**Next: Step 5.** On `diag_narrow` every $|s_k|$ was about 0.1, and on `shift_n` every one was exactly 1 until the last step. Step 5 asks what decides that: how GMRES's residual is a polynomial in $A$ applied to $b$, and how the eigenvalues of $A$ make that polynomial easy or hard to keep small.